* Importar y mostrar el dataframe

In [1]:
import kagglehub
import os

# Download latest version
path = kagglehub.dataset_download("lava18/google-play-store-apps")

print("Path to dataset files:", path)
print("Contents of the downloaded directory:")
for root, dirs, files in os.walk(path):
    for name in files:
        print(os.path.join(root, name))
    for name in dirs:
        print(os.path.join(root, name))

Using Colab cache for faster access to the 'google-play-store-apps' dataset.
Path to dataset files: /kaggle/input/google-play-store-apps
Contents of the downloaded directory:
/kaggle/input/google-play-store-apps/googleplaystore.csv
/kaggle/input/google-play-store-apps/license.txt
/kaggle/input/google-play-store-apps/googleplaystore_user_reviews.csv


In [2]:
import pandas as pd

csv_file_name = 'googleplaystore.csv'
csv_file_path = os.path.join(path, csv_file_name)

df = pd.read_csv(csv_file_path)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10841 entries, 0 to 10840
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   App             10841 non-null  object 
 1   Category        10841 non-null  object 
 2   Rating          9367 non-null   float64
 3   Reviews         10841 non-null  object 
 4   Size            10841 non-null  object 
 5   Installs        10841 non-null  object 
 6   Type            10840 non-null  object 
 7   Price           10841 non-null  object 
 8   Content Rating  10840 non-null  object 
 9   Genres          10841 non-null  object 
 10  Last Updated    10841 non-null  object 
 11  Current Ver     10833 non-null  object 
 12  Android Ver     10838 non-null  object 
dtypes: float64(1), object(12)
memory usage: 1.1+ MB


* Mostrar información sobre el dataframe

In [3]:
print("Información de los datos")
print(f"Datos duplicados: {df.duplicated().sum()}")
print(f"Datos faltantes: \n{df.isna().sum()}")

Información de los datos
Datos duplicados: 483
Datos faltantes: 
App                  0
Category             0
Rating            1474
Reviews              0
Size                 0
Installs             0
Type                 1
Price                0
Content Rating       1
Genres               0
Last Updated         0
Current Ver          8
Android Ver          3
dtype: int64


In [4]:
#Verificar outliers de rating
out_of_range_rating = df[(df['Rating'] < 1) | (df['Rating'] > 5)]
if not out_of_range_rating.empty:
    print(f"\nValores de 'Rating' fuera del rango (1-5): {len(out_of_range_rating)} filas")
    display(out_of_range_rating[['App', 'Rating']].head())
else:
    print("\nTodos los valores de 'Rating' están dentro del rango (1-5).")


Valores de 'Rating' fuera del rango (1-5): 1 filas


,App,Rating
10472,Life Made WI-Fi Touchscreen Photo Frame,19.0


In [5]:
from numpy import nan
# Verificar outliers de type
out_of_range_type = df[(df['Type'] != "Free") & (df['Type'] != "Paid") & (df['Type'] != nan)]
if not out_of_range_type.empty:
    print(f"\nValores de 'Type' no válidos: {len(out_of_range_type)} filas")
    display(out_of_range_type[['App','Type']].head())
else:
    print("\nTodos los valores de 'Type' son válidos.")


Valores de 'Type' no válidos: 2 filas


,App,Type
9148,Command & Conquer: Rivals,NaN
10472,Life Made WI-Fi Touchscreen Photo Frame,0


In [6]:
#Verificar outliers de Last Updated

# Intentar convertir a datetime en una copia temporal para detectar fechas inválidas
converted_dates_temp = pd.to_datetime(df['Last Updated'], errors='coerce')

# Filtrar filas donde la conversión falló (NaT)
invalid_dates = df[converted_dates_temp.isna()]

# Mostrar resultados
if not invalid_dates.empty:
    print(f"\nValores inválidos en 'Last Updated': {len(invalid_dates)} filas")
    display(invalid_dates[['App', 'Last Updated']].head())
else:
    print("\nTodos los valores de 'Last Updated' parecen ser fechas válidas.")


Valores inválidos en 'Last Updated': 1 filas


,App,Last Updated
10472,Life Made WI-Fi Touchscreen Photo Frame,1.0.19


* Creamos copia para empezar a limpiar

In [7]:
df_clean = df.copy()

* Eliminamos datos duplicados

In [8]:
print(f"Dataframe antes de limpiar: {df_clean.duplicated().sum()}")
df_clean = df.drop_duplicates()
print(f"Dataframe después de limpiar: {df_clean.duplicated().sum()}")

Dataframe antes de limpiar: 483
Dataframe después de limpiar: 0


* Convertirmos outliers a nan, para luego reemplazarlos

In [9]:
df_clean.loc[(df_clean['Rating']< 1) | (df_clean['Rating'] > 5), 'Rating'] = nan

out_of_range_rating = df_clean[(df_clean['Rating'] < 1) | (df_clean['Rating'] > 5)]
if not out_of_range_rating.empty:
    print(f"\nValores de 'Rating' fuera del rango (1-5): {len(out_of_range_rating)} filas")
    display(out_of_range_rating[['App', 'Rating']].head())
else:
    print("\nTodos los valores de 'Rating' están dentro del rango (1-5).")

#print(df_clean.loc[10472])


Todos los valores de 'Rating' están dentro del rango (1-5).


In [10]:
df_clean.loc[(df_clean['Type'] != "Free") & (df_clean['Type'] != "Paid")  & df_clean['Type'].notna(), 'Type' ] = nan

out_of_range_type = df_clean[(df_clean['Type'] != "Free") & (df_clean['Type'] != "Paid") & df_clean['Type'].notna()]
if not out_of_range_type.empty:
    print(f"\nValores de 'Type' no válidos: {len(out_of_range_type)} filas")
    display(out_of_range_type[['App','Type']].head())
else:
    print("\nTodos los valores de 'Type' son válidos.")


Todos los valores de 'Type' son válidos.


In [11]:
# Convertir y limpiar en el DataFrame real
df_clean.loc[:, 'Last Updated'] = pd.to_datetime(
    df_clean['Last Updated'],
    errors='coerce'
)

# Detectar outliers (ya convertidos a NaT)
invalid_dates = df_clean[df_clean['Last Updated'].isna()]

# Mostrar solo los problemáticos
if invalid_dates.empty:
    print(f"\nValores de 'Last Updated' inválidos: {len(invalid_dates)} filas")
    display(invalid_dates[['App', 'Last Updated']].head())
else:
    print("\nTodos los valores de 'Last Updated' son válidos.")


Todos los valores de 'Last Updated' son válidos.


* Rellenar datos faltantes

--  Para rellenar rating, usamos el promedio de ratings y rellenamos.

In [12]:
promedio_rating = round(df_clean['Rating'].mean(),1)
print(f"El promedio de los ratings es: {promedio_rating:.2f}")

df_clean['Rating'] = df_clean['Rating'].fillna(promedio_rating)

print(f"\nDatos faltantes después de rellenar 'Rating': \n{df_clean.isna().sum()}")

El promedio de los ratings es: 4.20

Datos faltantes después de rellenar 'Rating': 
App               0
Category          0
Rating            0
Reviews           0
Size              0
Installs          0
Type              2
Price             0
Content Rating    1
Genres            0
Last Updated      1
Current Ver       8
Android Ver       3
dtype: int64


/tmp/ipykernel_2112/3083353389.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['Rating'] = df_clean['Rating'].fillna(promedio_rating)


-- Para rellenar Type, tomaremos en cuenta el precio de la aplicación, si es 0 Type es 'Free', y si es mayor a 0 Type es 'Paid'.

In [13]:
# Rellenar valores 'Type' donde son NaN
# Si 'Price' es '0', rellenar 'Type' como 'Free'
df_clean.loc[(df_clean['Type'].isna()) & (df_clean['Price'] == '0'), 'Type'] = 'Free'

# Si 'Price' no es '0', rellenar 'Type' como 'Paid'
df_clean.loc[(df_clean['Type'].isna()) & (df_clean['Price'] != '0'), 'Type'] = 'Paid'

print(f"\nDatos faltantes después de rellenar 'Type': \n{df_clean.isna().sum()}")


Datos faltantes después de rellenar 'Type': 
App               0
Category          0
Rating            0
Reviews           0
Size              0
Installs          0
Type              0
Price             0
Content Rating    1
Genres            0
Last Updated      1
Current Ver       8
Android Ver       3
dtype: int64


-- Para rellenar Content Rating, usamos el una función random, para seleccionar de manera aleaotoria un tipo de dato de los ya existentes.

In [14]:
import random
# Obtener los valores que ya están
existing_content_rating_values = df_clean['Content Rating'].dropna().unique()

# Rellenar
df_clean['Content Rating'] = df_clean['Content Rating'].fillna(pd.Series(random.choices(existing_content_rating_values, k=len(df_clean)), index=df_clean.index))

print(f"Datos faltantes después de rellenar 'Content Rating': \n{df_clean.isna().sum()}")

Datos faltantes después de rellenar 'Content Rating': 
App               0
Category          0
Rating            0
Reviews           0
Size              0
Installs          0
Type              0
Price             0
Content Rating    0
Genres            0
Last Updated      1
Current Ver       8
Android Ver       3
dtype: int64


/tmp/ipykernel_2112/3248221548.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['Content Rating'] = df_clean['Content Rating'].fillna(pd.Series(random.choices(existing_content_rating_values, k=len(df_clean)), index=df_clean.index))


-- Para rellenar Last Update, usamos la fecha actual

In [15]:
from datetime import datetime

current_date_str = datetime.now().strftime('%Y-%m-%d')

df_clean.loc[(df_clean['Last Updated'].isna()), 'Last Updated'] = current_date_str

print(f"\nDatos faltantes después de rellenar 'Current Ver': \n{df_clean.isna().sum()}")


Datos faltantes después de rellenar 'Current Ver': 
App               0
Category          0
Rating            0
Reviews           0
Size              0
Installs          0
Type              0
Price             0
Content Rating    0
Genres            0
Last Updated      0
Current Ver       8
Android Ver       3
dtype: int64


-- Para rellenar Current Ver, damos un valor genérico ya existente 'Varies with device'

In [16]:
df_clean.loc[(df_clean['Current Ver'].isna()), 'Current Ver'] = 'Varies with device'

print(f"\nDatos faltantes después de rellenar 'Current Ver': \n{df_clean.isna().sum()}")


Datos faltantes después de rellenar 'Current Ver': 
App               0
Category          0
Rating            0
Reviews           0
Size              0
Installs          0
Type              0
Price             0
Content Rating    0
Genres            0
Last Updated      0
Current Ver       0
Android Ver       3
dtype: int64


-- Para rellenar Android Ver, damos un valor genérico ya existente '4.0 and up'



In [17]:
df_clean.loc[(df_clean['Android Ver'].isna()), 'Android Ver'] = '4.0 and up'

print(f"\nDatos faltantes después de rellenar 'Android Ver': \n{df_clean.isna().sum()}")


Datos faltantes después de rellenar 'Android Ver': 
App               0
Category          0
Rating            0
Reviews           0
Size              0
Installs          0
Type              0
Price             0
Content Rating    0
Genres            0
Last Updated      0
Current Ver       0
Android Ver       0
dtype: int64


- Transformaciones

In [18]:
print(df_clean.loc[10472])
df_clean.drop(index = 10472, inplace = True)

App               Life Made WI-Fi Touchscreen Photo Frame
Category                                              1.9
Rating                                                4.2
Reviews                                              3.0M
Size                                               1,000+
Installs                                             Free
Type                                                 Paid
Price                                            Everyone
Content Rating                                       Teen
Genres                                  February 11, 2018
Last Updated                                   2026-05-14
Current Ver                                    4.0 and up
Android Ver                                    4.0 and up
Name: 10472, dtype: object


/tmp/ipykernel_2112/852326978.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean.drop(index = 10472, inplace = True)


In [19]:
# Limpiar y convertir 'Installs' a float
df_clean['Installs'] = (
    df_clean['Installs']
    .astype(str)
    .str.replace(r'[^0-9]', '', regex=True)   # elimina TODO lo que no sea número
    .pipe(pd.to_numeric)     # convierte a número
    .astype('float64')                        # asegura float
)

# Verificación
print(df_clean[['Installs']].head())
print(f"El tipo de la columna 'Installs' es: {df_clean['Installs'].dtype}")

     Installs
0     10000.0
1    500000.0
2   5000000.0
3  50000000.0
4    100000.0
El tipo de la columna 'Installs' es: float64


/tmp/ipykernel_2112/638241193.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['Installs'] = (


In [20]:
# Limpiar y convertir 'Price' a float
df_clean['Price'] = (
    df_clean['Price']
    .astype(str)
    .str.replace(r'[^0-9.]', '', regex=True)  # deja solo números y punto decimal
    .pipe(pd.to_numeric)     # convierte a número
    .astype('float64')                        # asegura float
)

# Verificación
print(df_clean[['Price']].sample(5))
print(f"El tipo de la columna 'Price' es: {df_clean['Price'].dtype}")

      Price
8300   0.00
4301   5.99
5583   0.00
1135   0.00
2849   0.00
El tipo de la columna 'Price' es: float64


/tmp/ipykernel_2112/3433542061.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['Price'] = (


In [21]:
# Limpiar y convertir la columna 'Size' a formato numérico
def clean_size(size):
    if pd.isna(size) or size == 'Varies with device':
        return None
    size = str(size).strip() # Eliminar espacios en blanco
    size = size.replace(',', '') # Eliminar comas, e.g., '1,000+'

    if size.endswith('M'):
        return float(size.replace('M', '')) * 1000000
    elif size.endswith('k'):
        return float(size.replace('k', '')) * 1000
    elif size.endswith('+'): # e.g., '1000+'
        return float(size.replace('+', ''))
    try:
        return float(size)
    except ValueError:
        return None # Retorna None si no se puede convertir por otras razones

# 1. Aplicar la función clean_size, creando una serie temporal.
temp_size_series = df_clean['Size'].apply(clean_size)

# 2. Convertir a tipo numérico, coercing errors (None se convierte a NaN).
temp_size_series = pd.to_numeric(temp_size_series, errors='coerce')

# 3. Calcular la mediana.
median_size = temp_size_series.median()

# 4. Rellenar los valores NaN con la mediana.
temp_size_series = temp_size_series.fillna(median_size)

# 5. Asignar la serie completamente procesada de nuevo a la columna 'Size' del DataFrame,
#    y luego forzar explícitamente el tipo a float64 en la columna del DataFrame.
df_clean['Size'] = temp_size_series

df_clean['Size'] = df_clean['Size'].astype('float64')

print(f"El tipo de la columna 'Size' después de la transformación es: {df_clean['Size'].dtype}")
print("Primeras 5 filas de 'Size' después de la transformación:")
print(df_clean['Size'].head())

El tipo de la columna 'Size' después de la transformación es: float64
Primeras 5 filas de 'Size' después de la transformación:
0    19000000.0
1    14000000.0
2     8700000.0
3    25000000.0
4     2800000.0
Name: Size, dtype: float64


/tmp/ipykernel_2112/3632766395.py:33: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['Size'] = temp_size_series
/tmp/ipykernel_2112/3632766395.py:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['Size'] = df_clean['Size'].astype('float64')


In [22]:
df_clean['Reviews'] = (
    df_clean['Reviews']
    .astype(str)
    .str.strip()                              # eliminar espacios invisibles
    .replace('', '0')                         # strings vacíos → 0
    .str.replace(r'[^0-9]', '', regex=True)   # solo números
    .pipe(pd.to_numeric)     # a número
)

# Forzar tipo al final (MUY IMPORTANTE)
df_clean['Reviews'] = df_clean['Reviews'].astype('float64')

# Verificación
print(df_clean[['Reviews']].head())
print(f"El tipo de la columna 'Reviews' es: {df_clean['Reviews'].dtype}")

    Reviews
0     159.0
1     967.0
2   87510.0
3  215644.0
4     967.0
El tipo de la columna 'Reviews' es: float64


/tmp/ipykernel_2112/1169961794.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['Reviews'] = (
/tmp/ipykernel_2112/1169961794.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['Reviews'] = df_clean['Reviews'].astype('float64')


In [23]:
from datetime import datetime

#Convertir la columna 'Last Updated' en datetime
converted_dates = pd.to_datetime(df_clean['Last Updated'], format='mixed')

df_clean['Last Updated'] = converted_dates

#Verificar que la columna haya cambiado
print(f"Dtype of 'Last Updated' after conversion: {df_clean['Last Updated'].dtype}")

#Obtener dia actual, para el calculo de dias
current_date_ts = pd.Timestamp(datetime.now().date())

#Calcular la cantidad de dias desde la ultima actualizacion
df_clean.loc[:, 'Days Last Update'] = (current_date_ts - df_clean['Last Updated']).dt.days

print(df_clean[['Last Updated', 'Days Last Update']].head())
print(f"El tipo de la columna 'Days Last Update' es: {df_clean['Days Last Update'].dtype}")

Dtype of 'Last Updated' after conversion: datetime64[ns]
  Last Updated  Days Last Update
0   2018-01-07              3049
1   2018-01-15              3041
2   2018-08-01              2843
3   2018-06-08              2897
4   2018-06-20              2885
El tipo de la columna 'Days Last Update' es: int64


/tmp/ipykernel_2112/2668994132.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['Last Updated'] = converted_dates
/tmp/ipykernel_2112/2668994132.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean.loc[:, 'Days Last Update'] = (current_date_ts - df_clean['Last Updated']).dt.days


In [24]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10357 entries, 0 to 10840
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   App               10357 non-null  object        
 1   Category          10357 non-null  object        
 2   Rating            10357 non-null  float64       
 3   Reviews           10357 non-null  float64       
 4   Size              10357 non-null  float64       
 5   Installs          10357 non-null  float64       
 6   Type              10357 non-null  object        
 7   Price             10357 non-null  float64       
 8   Content Rating    10357 non-null  object        
 9   Genres            10357 non-null  object        
 10  Last Updated      10357 non-null  datetime64[ns]
 11  Current Ver       10357 non-null  object        
 12  Android Ver       10357 non-null  object        
 13  Days Last Update  10357 non-null  int64         
dtypes: datetime64[ns](1), float

* Pipeline

In [25]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Definir las columnas numéricas y categóricas para el ColumnTransformer
numeric_features = ['Rating', 'Reviews', 'Size', 'Installs', 'Price', 'Days Last Update']
categorical_features = ['Category', 'Type', 'Content Rating', 'Genres', 'Current Ver', 'Android Ver']

# Crear el ColumnTransformer
pipeline = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

# Aplicar las transformaciones a X
X_transf = pipeline.fit_transform(df_clean.drop(['App', 'Last Updated'], axis=1)) # Excluir 'App' y 'Last Updated'
print("Forma de X_transf:", X_transf.shape)
print("Primeros elementos de X_transf (representación sparse):")
print(X_transf[:2])

Forma de X_transf: (10357, 3030)
Primeros elementos de X_transf (representación sparse):
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 24 stored elements and shape (2, 3030)>
  Coords	Values
  (0, 0)	-0.1851009037091209
  (0, 1)	-0.15046296898474376
  (0, 2)	-0.05074846506919142
  (0, 3)	-0.17632753190442066
  (0, 4)	-0.06332538377467443
  (0, 5)	-0.13448629262311967
  (0, 6)	1.0
  (0, 39)	1.0
  (0, 42)	1.0
  (0, 56)	1.0
  (0, 284)	1.0
  (0, 3012)	1.0
  (1, 0)	-0.598311391846927
  (1, 1)	-0.1501633376991226
  (1, 2)	-0.2886320709327555
  (1, 3)	-0.17022052315893652
  (1, 4)	-0.06332538377467443
  (1, 5)	-0.15455609706103032
  (1, 6)	1.0
  (1, 39)	1.0
  (1, 42)	1.0
  (1, 59)	1.0
  (1, 1184)	1.0
  (1, 3012)	1.0


### Evaluación 2


### División de los datos en conjuntos de entrenamiento y prueba

In [26]:
df_trans = df_clean.copy()

mediana_rating = df_trans['Rating'].median()

df_trans['Quality Status'] = '' # Initialize the column
df_trans.loc[df_trans['Rating'] >= mediana_rating, 'Quality Status'] = 'Good'
df_trans.loc[df_trans['Rating'] < mediana_rating, 'Quality Status'] = 'Bad'

In [27]:
df_trans.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10357 entries, 0 to 10840
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   App               10357 non-null  object        
 1   Category          10357 non-null  object        
 2   Rating            10357 non-null  float64       
 3   Reviews           10357 non-null  float64       
 4   Size              10357 non-null  float64       
 5   Installs          10357 non-null  float64       
 6   Type              10357 non-null  object        
 7   Price             10357 non-null  float64       
 8   Content Rating    10357 non-null  object        
 9   Genres            10357 non-null  object        
 10  Last Updated      10357 non-null  datetime64[ns]
 11  Current Ver       10357 non-null  object        
 12  Android Ver       10357 non-null  object        
 13  Days Last Update  10357 non-null  int64         
 14  Quality Status    10357 non

In [28]:
# Nueva copia para trabajar la ev 2
#rating como regresion / columan nueva sobre rating con meadin, 1 bien, 0 mal
df_trans = df_trans.drop(['App'], axis=1)
df_trans = df_trans.drop(['Category'], axis=1)
df_trans = df_trans.drop(['Rating'], axis=1)
df_trans = df_trans.drop(['Type'], axis=1)
df_trans = df_trans.drop(['Content Rating'], axis=1)
df_trans = df_trans.drop(['Genres'], axis=1)
df_trans = df_trans.drop(['Last Updated'], axis=1)
df_trans = df_trans.drop(['Current Ver'], axis=1)
df_trans = df_trans.drop(['Android Ver'], axis=1)

display(df_trans)

,Reviews,Size,Installs,Price,Days Last Update,Quality Status
0,159.0,19000000.0,10000.0,0.0,3049,Bad
1,967.0,14000000.0,500000.0,0.0,3041,Bad
2,87510.0,8700000.0,5000000.0,0.0,2843,Good
3,215644.0,25000000.0,50000000.0,0.0,2897,Good
4,967.0,2800000.0,100000.0,0.0,2885,Good
...,...,...,...,...,...,...
10836,38.0,53000000.0,5000.0,0.0,3215,Good
10837,4.0,3600000.0,100.0,0.0,2869,Good
10838,3.0,9500000.0,1000.0,0.0,3401,Good
10839,114.0,13000000.0,1000.0,0.0,4133,Good


In [29]:
X = df_trans.drop('Quality Status', axis = 1)
y = df_trans['Quality Status']

In [30]:
from sklearn.model_selection import train_test_split

# Dividir los datos en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X_transf, y, test_size=0.2, random_state=42)

print(f"Forma de X_train: {X_train.shape}")
print(f"Forma de X_test: {X_test.shape}")
print(f"Forma de y_train: {y_train.shape}")
print(f"Forma de y_test: {y_test.shape}")

Forma de X_train: (8285, 3030)
Forma de X_test: (2072, 3030)
Forma de y_train: (8285,)
Forma de y_test: (2072,)


In [31]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

# Entrenar modelo
clf = LogisticRegression()
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
# Evaluar
print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))

Accuracy: 1.0
F1 Score: 1.0


##GridSearchCV

In [32]:
param_grid = {
    'n_estimators': [ 100, 150, 200, 250], # Puedes probar más estimadores
    'max_depth': [4, 8, 12, 16], # Ampliar el rango y añadir None
    'max_features': ['sqrt', 'log2'], # Incluir para explorar diferentes subconjuntos de características
    'criterion': ['gini', 'entropy'] # Evaluar ambos criterios
}

In [33]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier

clf = GridSearchCV(RandomForestClassifier(), param_grid, cv = 3, n_jobs=-1)
clf.fit(X_train,y_train)

GridSearchCV(cv=3, estimator=RandomForestClassifier(), n_jobs=-1,
             param_grid={'criterion': ['gini', 'entropy'],
                         'max_depth': [4, 8, 12, 16],
                         'max_features': ['sqrt', 'log2'],
                         'n_estimators': [100, 150, 200, 250]})

In [34]:
print("Mejores parámetros:", clf.best_params_)

Mejores parámetros: {'criterion': 'gini', 'max_depth': 16, 'max_features': 'sqrt', 'n_estimators': 200}


##RANDOMIZED SEARCH

In [35]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier

# param_grid ya debería estar definido

clf_random = RandomizedSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=3, n_iter=6, scoring='accuracy', random_state=42, n_jobs=-1)
clf_random.fit(X_train, y_train)
print("Mejores parámetros para RandomizedSearchCV:", clf_random.best_params_)
print("Mejor score de RandomizedSearchCV:", clf_random.best_score_)

Mejores parámetros para RandomizedSearchCV: {'n_estimators': 200, 'max_features': 'sqrt', 'max_depth': 16, 'criterion': 'entropy'}
Mejor score de RandomizedSearchCV: 0.9095844039198789


4. Evaluación del modelo

o Evaluar el modelo utilizando las métricas apropiadas (accuracy, F1 score, AUC, R- cuadrado, RMSE, etc.).

o Comparar los resultados obtenidos con otros modelos (si aplica) para determinar el mejor rendimiento.



5. Visualización de los resultados

o Crear visualizaciones claras que ayuden a entender el rendimiento del modelo, la distribución de las variables, y las decisiones tomadas durante el análisis.

o Incluir gráficos de desempeño, distribuciones de datos, y comparación de resultados.